<a href="https://colab.research.google.com/github/andersonmoraix/analisedeconjuntura_anotacoes/blob/main/aula01_extraindodadosviaAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas

In [ ]:
# Instala bibliotecas usando o pip
!pip install python-bcb # BCB
!pip install ipeadatapy # IPEADATA
!pip install sidrapy    # SIDRA/IBGE
!pip install dbnomics   # OCDE e FMI (outros também)
!pip install wbgapi     # Banco Mundial

# Importa bibliotecas/módulos
from bcb import sgs, Expectativas
from dbnomics import fetch_series_by_api_link
from datetime import date
import pandas as pd
import ipeadatapy as ipea
import sidrapy as sidra
import wbgapi as wb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.3 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 8.5.0
    Uninstalling tenacity-8.5.0:
      Successfully uninstalled tenacity-8.5.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 wh

# Dados do BCB

## SGS

In [ ]:
# Site: https://www3.bcb.gov.br/sgspub/
# Coletar dados da SELIC no SGS/BCB
dados_sgs = sgs.get(
    codes = 432,
    start = "2020-01-01",
    end = date.today()
)
dados_sgs

,432
Date,
2020-01-01,4.5
2020-01-02,4.5
2020-01-03,4.5
2020-01-04,4.5
2020-01-05,4.5
...,...
2025-08-26,15.0
2025-08-27,15.0
2025-08-28,15.0


In [ ]:
# Coletar múltiplas séries do SGS
dados_sgs = sgs.get(
    codes = {"Dólar": 3698, "IBC-Br": 24363, "Resultado Primário": 5793},
    start = "2020-01-01",
    end = date.today()
)
dados_sgs.tail()

,Dólar,IBC-Br,Resultado Primário
Date,,,
2025-03-01,5.7468,113.80032,0.11
2025-04-01,5.7837,112.23595,0.05
2025-05-01,5.6674,108.40943,-0.20
2025-06-01,5.5471,107.26420,-0.15
2025-07-01,5.5285,NaN,0.22


In [ ]:
# Tratamento de dados
dados_sgs.rename_axis("data").dropna().tail()

,Dólar,IBC-Br,Resultado Primário
data,,,
2025-02-01,5.7656,106.80038,0.13
2025-03-01,5.7468,113.80032,0.11
2025-04-01,5.7837,112.23595,0.05
2025-05-01,5.6674,108.40943,-0.20
2025-06-01,5.5471,107.26420,-0.15


In [ ]:
(dados_sgs.rename_axis("data") # formato wide p/ long
    .dropna()
    .reset_index()
    .melt(
        id_vars = "data",
        value_vars = ["Dólar", "IBC-Br", "Resultado Primário"],
        var_name = "variavel"
        )
    .rename(columns = {"value": "valor"})
    )

,data,variavel,valor
0,2020-01-01,Dólar,4.1495
1,2020-02-01,Dólar,4.3410
2,2020-03-01,Dólar,4.8839
3,2020-04-01,Dólar,5.3256
4,2020-05-01,Dólar,5.6434
...,...,...,...
193,2025-02-01,Resultado Primário,0.1300
194,2025-03-01,Resultado Primário,0.1100
195,2025-04-01,Resultado Primário,0.0500
196,2025-05-01,Resultado Primário,-0.2000


## Expectativas

In [ ]:
# Site: https://www3.bcb.gov.br/expectativas2/#/consultaSeriesEstatisticas
# Coletar dados de expectativas de mercado

# Conexão com a API
exp = Expectativas().get_endpoint("ExpectativasMercadoAnuais")

# Extração dos dados
dados_focus = (
    exp.query()
    .filter(exp.Indicador == "IPCA")
    .filter(exp.Data >= "2020-01-01")
    .collect()
    )

dados_focus

,Indicador,IndicadorDetalhe,Data,DataReferencia,Media,Mediana,DesvioPadrao,Minimo,Maximo,numeroRespondentes,baseCalculo
0,IPCA,None,2020-01-02,2019,4.0500,4.1100,0.2000,3.4300,4.3600,121,0
1,IPCA,None,2020-01-02,2020,3.5900,3.6000,0.2500,2.4600,4.2300,119,0
2,IPCA,None,2020-01-02,2021,3.7400,3.7500,0.1300,3.3000,4.1000,102,0
3,IPCA,None,2020-01-02,2022,3.5600,3.5000,0.1600,3.0000,4.1200,93,0
4,IPCA,None,2020-01-02,2023,3.5000,3.5000,0.2100,3.0000,4.1100,78,0
...,...,...,...,...,...,...,...,...,...,...,...
14155,IPCA,None,2025-08-22,2025,4.8520,4.8400,0.2051,4.4731,5.9853,111,1
14156,IPCA,None,2025-08-22,2026,4.3209,4.3201,0.3181,3.3800,5.7000,110,1
14157,IPCA,None,2025-08-22,2027,3.8951,3.9400,0.4450,2.9500,5.2300,99,1
14158,IPCA,None,2025-08-22,2028,3.7565,3.7000,0.4797,3.0000,5.0000,91,1


In [ ]:
# Extração com tratamentos de dados
dados_focus = (
    exp.query()
    .filter(exp.Indicador == "IPCA")
    .filter(exp.Data >= "2020-01-01")
    .filter(exp.baseCalculo == 0)
    .select(exp.Data, exp.DataReferencia, exp.Mediana)
    .collect()
    )

dados_focus

,Data,DataReferencia,Mediana
0,2020-01-02,2019,4.1100
1,2020-01-02,2020,3.6000
2,2020-01-02,2021,3.7500
3,2020-01-02,2022,3.5000
4,2020-01-02,2023,3.5000
...,...,...,...
7075,2025-08-22,2025,4.8639
7076,2025-08-22,2026,4.3274
7077,2025-08-22,2027,3.9650
7078,2025-08-22,2028,3.8000


In [ ]:
# Agregar expectativa por mês
dados_focus.rename(
    columns = {"Data": "data", "DataReferencia": "data_ref", "Mediana": "mediana"},
    inplace = True
    )
dados_focus

,data,data_ref,mediana
0,2020-01-02,2019,4.1100
1,2020-01-02,2020,3.6000
2,2020-01-02,2021,3.7500
3,2020-01-02,2022,3.5000
4,2020-01-02,2023,3.5000
...,...,...,...
7075,2025-08-22,2025,4.8639
7076,2025-08-22,2026,4.3274
7077,2025-08-22,2027,3.9650
7078,2025-08-22,2028,3.8000


In [ ]:
dados_focus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7080 entries, 0 to 7079
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   data      7080 non-null   datetime64[ns]
 1   data_ref  7080 non-null   object        
 2   mediana   7080 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 166.1+ KB


In [ ]:
# Converter coluna data e finalizar agregação
dados_focus["data"] = pd.to_datetime(dados_focus["data"])
dados_focus["ano_mes"] = dados_focus["data"].dt.strftime("%Y/%m")
dados_focus.groupby(by = ["data_ref", "ano_mes"])["mediana"].mean()

data_ref  ano_mes
2019      2020/01    4.148333
2020      2020/01    3.543182
          2020/02    3.261667
          2020/03    3.087727
          2020/04    2.385000
                       ...   
2029      2025/04    3.626250
          2025/05    3.727024
          2025/06    3.768375
          2025/07    3.665761
          2025/08    3.551381
Name: mediana, Length: 346, dtype: float64

# Dados do IPEADATA

In [ ]:
# Site: http://www.ipeadata.gov.br/
# Extrair tabela com todas séries e códigos disponíveis
series_ipeadata = ipea.metadata()
series_ipeadata.tail()

,CODE,NAME,COMMENT,LAST UPDATE,BIG THEME,SOURCE ACRONYM,SOURCE,SOURCE URL,FREQUENCY,MEASURE,UNIT,SERIES STATUS,THEME CODE,COUNTRY,NUMERICA
2801,PNADCT_TXPARTCUF_SI,Taxa de participação - sem instrução ou equiva...,"Taxa de participação na força de trabalho, na ...",2025-08-15T11:12:10.65-03:00,Social,IBGE/PNAD Contínua,Instituto Brasileiro de Geografia e Estatístic...,http://www.ibge.gov.br/home/estatistica/indica...,Trimestral,(%),None,None,110,None,True
2802,PNAD_IAGRV,Domicílios com insegurança alimentar grave,Distribuição percentual dos domicílios de acor...,2024-06-20T10:44:01.677-03:00,Social,IBGE/PNAD Contínua,Instituto Brasileiro de Geografia e Estatístic...,http://www.ibge.gov.br/home/estatistica/indica...,Decenal,(%),None,None,111,None,True
2803,PNAD_IALEV,Domicílios com insegurança alimentar leve,Distribuição percentual dos domicílios de acor...,2024-06-20T10:44:01.68-03:00,Social,IBGE/PNAD Contínua,Instituto Brasileiro de Geografia e Estatístic...,http://www.ibge.gov.br/home/estatistica/indica...,Decenal,(%),None,None,111,None,True
2804,PNAD_IAMOD,Domicílios com insegurança alimentar moderada,Distribuição percentual dos domicílios de acor...,2024-06-20T10:44:01.683-03:00,Social,IBGE/PNAD Contínua,Instituto Brasileiro de Geografia e Estatístic...,http://www.ibge.gov.br/home/estatistica/indica...,Decenal,(%),None,None,111,None,True
2805,PNAD_SATOT,Domicílios com segurança alimentar,Distribuição percentual dos domicílios de acor...,2024-06-20T10:44:01.69-03:00,Social,IBGE/PNAD Contínua,Instituto Brasileiro de Geografia e Estatístic...,http://www.ibge.gov.br/home/estatistica/indica...,Decenal,(%),None,None,111,None,True


In [ ]:
# Filtrar séries com o termo "caged"
series_ipeadata[series_ipeadata.CODE.str.contains("caged|CAGED|Caged")]

,CODE,NAME,COMMENT,LAST UPDATE,BIG THEME,SOURCE ACRONYM,SOURCE,SOURCE URL,FREQUENCY,MEASURE,UNIT,SERIES STATUS,THEME CODE,COUNTRY,NUMERICA
1191,CAGED12_ADMIS,Empregados - admissões - INATIVA,Os dados referem-se ao total de admissões de ...,2020-05-27T14:48:01-03:00,Macroeconômico,MTE/Caged,"Ministério do Trabalho e Emprego, Cadastro Ger...",www.mte.gov.br,Mensal,Pessoa,None,I,12,BRA,True
1192,CAGED12_ADMISN12,Empregados - admissões - sem ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True
1193,CAGED12_ADMISNAJU12,Empregados - admissões - com ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True
1194,CAGED12_DESLIG,Empregados - demissões - INATIVA,Os dados referem-se ao total de dispensa de e...,2020-05-27T14:48:00-03:00,Macroeconômico,MTE/Caged,"Ministério do Trabalho e Emprego, Cadastro Ger...",www.mte.gov.br,Mensal,Pessoa,None,I,12,BRA,True
1195,CAGED12_DESLIGN12,Empregados - demissões - sem ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True
1196,CAGED12_DESLIGNAJU12,Empregados - demissões - com ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True
1197,CAGED12_SALDO12,Empregados - saldo - INATIVA,O saldo refere-se a diferença entre o total de...,2020-05-27T14:48:00-03:00,Macroeconômico,MTE/Caged,"Ministério do Trabalho e Emprego, Cadastro Ger...",www.mte.gov.br,Mensal,Pessoa,None,I,12,BRA,True
1198,CAGED12_SALDON12,Empregados - saldo - sem ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True
1199,CAGED12_SALDONAJU12,Empregados - saldo - com ajuste - novo Caged,Para empregados consideram-se aqueles contrata...,2025-08-06T18:18:16.72-03:00,Macroeconômico,Min. Economia/SEPRT/Novo Caged,"Ministério da Economia, Secretaria Especial de...",http://pdet.mte.gov.br/novo-caged,Mensal,Pessoa,None,A,12,BRA,True


In [ ]:
# Coletar série de interesse usando o código
dados_ipeadata = ipea.timeseries("CAGED12_SALDON12")
dados_ipeadata.tail()

,CODE,RAW DATE,DAY,MONTH,YEAR,VALUE (Pessoa)
DATE,,,,,,
2025-02-01,CAGED12_SALDON12,2025-02-01T00:00:00-03:00,1,2,2025,431995.0
2025-03-01,CAGED12_SALDON12,2025-03-01T00:00:00-03:00,1,3,2025,71576.0
2025-04-01,CAGED12_SALDON12,2025-04-01T00:00:00-03:00,1,4,2025,257528.0
2025-05-01,CAGED12_SALDON12,2025-05-01T00:00:00-03:00,1,5,2025,148992.0
2025-06-01,CAGED12_SALDON12,2025-06-01T00:00:00-03:00,1,6,2025,166621.0


In [ ]:
# Tratamento de dados
(
    dados_ipeadata.reset_index()
    .rename(columns = {"DATE": "data", "VALUE (Pessoa)": "caged"})
    .filter(items = ["data", "caged"], axis = "columns")
    ).tail()

,data,caged
61,2025-02-01,431995.0
62,2025-03-01,71576.0
63,2025-04-01,257528.0
64,2025-05-01,148992.0
65,2025-06-01,166621.0


# Dados do Sidra/IBGE

In [ ]:
# Site: https://sidra.ibge.gov.br/ e https://apisidra.ibge.gov.br/
# Códigos de consulta (obtidos no site) com filtros na tabela 7060 (Sidra/IBGE)
dados_sidra = sidra.get_table(
    table_code = "7060",             # código da tabela de interesse
    territorial_level = "1",         # nível territorial
    ibge_territorial_code = "all",   # desagregações desse nível
    variable = "63",                 # variável = IPCA - Variação mensal (%)
    period = "all"                   # todas as observações
)

dados_sidra.head()

,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Brasil (Código),Brasil,Mês (Código),Mês,Variável (Código),Variável,"Geral, grupo, subgrupo, item e subitem (Código)","Geral, grupo, subgrupo, item e subitem"
1,1,Brasil,2,%,0.21,1,Brasil,202001,janeiro 2020,63,IPCA - Variação mensal,7169,Índice geral
2,1,Brasil,2,%,0.25,1,Brasil,202002,fevereiro 2020,63,IPCA - Variação mensal,7169,Índice geral
3,1,Brasil,2,%,0.07,1,Brasil,202003,março 2020,63,IPCA - Variação mensal,7169,Índice geral
4,1,Brasil,2,%,-0.31,1,Brasil,202004,abril 2020,63,IPCA - Variação mensal,7169,Índice geral


In [ ]:
# Tratamento de dados
(
    dados_sidra.rename(columns = dados_sidra.iloc[0])
    .rename(columns = {"Mês (Código)": "data", "Valor": "ipca"})#organizando colunas
    .drop(labels = 0, axis = "index")
    .filter(items = ["data", "ipca"], axis = "columns")
    .assign(data = lambda x: pd.to_datetime(arg = x.data, format = "%Y%m"))#organizando colunas , alterando os nomes dos itens das colunas
    ).tail()

,data,ipca
63,2025-03-01,0.56
64,2025-04-01,0.43
65,2025-05-01,0.26
66,2025-06-01,0.24
67,2025-07-01,0.26


# Dados da OECD

In [ ]:
# Site: https://stats.oecd.org/
# Banco de dados utilizado: https://db.nomics.world/

# Variável de interesse: Taxa de desemprego (harmonizada)
# Conjunto de dados: Labour Force Survey
# Link da tabela dessa variável na base do DBnomics:
cod_json = (
    "https://api.db.nomics.world/v22/series/OECD/MEI?dimensions=%7B%22" +
    "SUBJECT%22%3A%5B%22LRHUTTTT%22%5D%2C%22MEASURE%22%3A%5B%22STSA%22" +
    "%5D%2C%22FREQUENCY%22%3A%5B%22M%22%5D%7D&observations=1"
    )

# Coletar dados
dados_ocde = fetch_series_by_api_link(cod_json)

dados_ocde.head()

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,LOCATION,SUBJECT,MEASURE,FREQUENCY,Country,Subject,Measure,Frequency
0,monthly,OECD,MEI,Main Economic Indicators Publication,AUS.LRHUTTTT.STSA.M,Australia – Labour Force Survey - quarterly ra...,1978-02,1978-02-01,6.644695,6.644695,AUS,LRHUTTTT,STSA,M,Australia,Labour Force Survey - quarterly rates > Harmon...,"Level, rate or national currency, s.a.",Monthly
1,monthly,OECD,MEI,Main Economic Indicators Publication,AUS.LRHUTTTT.STSA.M,Australia – Labour Force Survey - quarterly ra...,1978-03,1978-03-01,6.302895,6.302895,AUS,LRHUTTTT,STSA,M,Australia,Labour Force Survey - quarterly rates > Harmon...,"Level, rate or national currency, s.a.",Monthly
2,monthly,OECD,MEI,Main Economic Indicators Publication,AUS.LRHUTTTT.STSA.M,Australia – Labour Force Survey - quarterly ra...,1978-04,1978-04-01,6.267631,6.267631,AUS,LRHUTTTT,STSA,M,Australia,Labour Force Survey - quarterly rates > Harmon...,"Level, rate or national currency, s.a.",Monthly
3,monthly,OECD,MEI,Main Economic Indicators Publication,AUS.LRHUTTTT.STSA.M,Australia – Labour Force Survey - quarterly ra...,1978-05,1978-05-01,6.209603,6.209603,AUS,LRHUTTTT,STSA,M,Australia,Labour Force Survey - quarterly rates > Harmon...,"Level, rate or national currency, s.a.",Monthly
4,monthly,OECD,MEI,Main Economic Indicators Publication,AUS.LRHUTTTT.STSA.M,Australia – Labour Force Survey - quarterly ra...,1978-06,1978-06-01,6.303555,6.303555,AUS,LRHUTTTT,STSA,M,Australia,Labour Force Survey - quarterly rates > Harmon...,"Level, rate or national currency, s.a.",Monthly


In [ ]:
# Tratamento de dados
dados_ocde.rename(
    columns = {"period": "data", "Country": "pais", "value": "valor"}
    ).filter(items = ["data", "pais", "valor"], axis = "columns")

,data,pais,valor
0,1978-02-01,Australia,6.644695
1,1978-03-01,Australia,6.302895
2,1978-04-01,Australia,6.267631
3,1978-05-01,Australia,6.209603
4,1978-06-01,Australia,6.303555
...,...,...,...
824,2023-09-01,United States,3.800000
825,2023-10-01,United States,3.800000
826,2023-11-01,United States,3.700000
827,2023-12-01,United States,3.700000


# Dados do FMI

In [ ]:
# Fonte: https://data.imf.org/
# Banco de dados utilizado: https://db.nomics.world/

# Variável de interesse: Taxa de Câmbio Real Efetiva (CPI, index)
# Conjunto de dados: International Financial Statistics (IFS)
# Link da tabela dessa variável do IFS na base do DBnomics:
cod_json = (
    "https://api.db.nomics.world/v22/series/IMF/IFS?dimensions=%7B%22INDICATOR" +
    "%22%3A%5B%22EREER_IX%22%5D%2C%22REF_AREA%22%3A%5B%22AU%22%2C%22CA%22%2C%2" +
    "2CL%22%5D%2C%22FREQ%22%3A%5B%22M%22%5D%7D&observations=1"
    )

# Coletar dados
dados_fmi = fetch_series_by_api_link(cod_json)

dados_fmi.head()

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,FREQ,REF_AREA,INDICATOR,Frequency,Reference Area,Indicator
0,monthly,IMF,IFS,International Financial Statistics (IFS),M.AU.EREER_IX,"Monthly – Australia – Exchange Rates, Real Eff...",1979-12,1979-12-01,90.735512,90.735512,M,AU,EREER_IX,Monthly,Australia,"Exchange Rates, Real Effective Exchange Rate b..."
1,monthly,IMF,IFS,International Financial Statistics (IFS),M.AU.EREER_IX,"Monthly – Australia – Exchange Rates, Real Eff...",1980-01,1980-01-01,90.509112,90.509112,M,AU,EREER_IX,Monthly,Australia,"Exchange Rates, Real Effective Exchange Rate b..."
2,monthly,IMF,IFS,International Financial Statistics (IFS),M.AU.EREER_IX,"Monthly – Australia – Exchange Rates, Real Eff...",1980-02,1980-02-01,90.418089,90.418089,M,AU,EREER_IX,Monthly,Australia,"Exchange Rates, Real Effective Exchange Rate b..."
3,monthly,IMF,IFS,International Financial Statistics (IFS),M.AU.EREER_IX,"Monthly – Australia – Exchange Rates, Real Eff...",1980-03,1980-03-01,91.461604,91.461604,M,AU,EREER_IX,Monthly,Australia,"Exchange Rates, Real Effective Exchange Rate b..."
4,monthly,IMF,IFS,International Financial Statistics (IFS),M.AU.EREER_IX,"Monthly – Australia – Exchange Rates, Real Eff...",1980-04,1980-04-01,92.179926,92.179926,M,AU,EREER_IX,Monthly,Australia,"Exchange Rates, Real Effective Exchange Rate b..."


In [ ]:
# Tratamento de dados
(
    dados_fmi.pivot(      # formato long p/ wide
        index = "period",
        columns = "Reference Area",
        values = "value"
        )
    .reset_index()
    .dropna()
    )

Reference Area,period,Australia,Canada,Chile
11,1979-12-01,90.735512,93.436111,149.810517
12,1980-01-01,90.509112,92.939362,151.507936
13,1980-02-01,90.418089,93.841047,154.573706
14,1980-03-01,91.461604,93.247621,160.603629
15,1980-04-01,92.179926,92.495853,164.188024
...,...,...,...,...
554,2025-03-01,88.731574,77.426583,87.565362
555,2025-04-01,87.707504,78.639083,84.204879
556,2025-05-01,88.980038,78.807742,85.967983
557,2025-06-01,88.978613,79.541691,85.255880


# Dados do Banco Mundial

In [ ]:
# Site: https://data.worldbank.org/
# Pesquisar códigos de variáveis sobre inflação (CPI)
wb.search(q = "inflation")

ID,Name,Field,Value
DC.ODA.TOTL.KD,,Statisticalconceptandmethodology,... The data is expressed in constant U.S. dollar prices to account for inflation in the donor's currency and the changes in exchange rates with the U.S. dollar....
FD.AST.PRVT.GD.ZS,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FD.RES.LIQU.AS.ZS,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FI.RES.TOTL.CD,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FI.RES.TOTL.DT.ZS,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FI.RES.TOTL.MO,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FI.RES.XGLD.CD,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FM.AST.CGOV.ZG.M3,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FM.AST.DOMO.ZG.M3,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."
FM.AST.DOMS.CN,,Developmentrelevance,"...these statistics to craft monetary policy, manage interest rates, and regulate inflation. For investors and market analysts, these figures provide a window into the..."


In [ ]:
# Coletar dados
dados_wb = wb.data.DataFrame(
    series = "FP.CPI.TOTL.ZG",
    economy = "all",
    time = range(2010, 2023),
    columns = "series",
    labels = True
    )
dados_wb

Country  Time  FP.CPI.TOTL.ZG
economy time                                                     
ZWE     YR2022                     Zimbabwe  2022      104.705171
        YR2021                     Zimbabwe  2021       98.546105
        YR2020                     Zimbabwe  2020      557.201817
        YR2019                     Zimbabwe  2019      255.304991
        YR2018                     Zimbabwe  2018       10.618866
...                                     ...   ...             ...
AFE     YR2014  Africa Eastern and Southern  2014        5.370290
        YR2013  Africa Eastern and Southern  2013        5.748831
        YR2012  Africa Eastern and Southern  2012        9.158708
        YR2011  Africa Eastern and Southern  2011        8.971206
        YR2010  Africa Eastern and Southern  2010        5.537538

[3458 rows x 3 columns]

In [ ]:
# Tratamento de dados
(
    dados_wb
    .reset_index()
    .rename(
        columns = {
            "Time": "data",
            "Country": "pais",
            "economy": "codigo",
            "FP.CPI.TOTL.ZG": "cpi"
            }
            )
    .filter(items = ["data", "pais", "codigo", "cpi"], axis = "columns")
    )

,data,pais,codigo,cpi
0,2022,Zimbabwe,ZWE,104.705171
1,2021,Zimbabwe,ZWE,98.546105
2,2020,Zimbabwe,ZWE,557.201817
3,2019,Zimbabwe,ZWE,255.304991
4,2018,Zimbabwe,ZWE,10.618866
...,...,...,...,...
3453,2014,Africa Eastern and Southern,AFE,5.370290
3454,2013,Africa Eastern and Southern,AFE,5.748831
3455,2012,Africa Eastern and Southern,AFE,9.158708
3456,2011,Africa Eastern and Southern,AFE,8.971206
